# Machine Learning 2026
# Assignment 1
# Question 3: Regression
**Max Marks: 40**

## Instructions

- Keep collaborations at high level discussions. Copying or plagiarism will be dealt with strictly.
- Rename the notebook with your roll number before submitting. If your roll number is `2024000`, the file must be named `2024000_q3.ipynb`.
- Write code only in the sections marked `# YOUR CODE HERE`. You can NOT write code anywhere else.
- Extract `Q3_DATA.zip` next to the notebook.
- Seed every random operation with your roll number.
- Do not modify any function or class definition outside a `# YOUR CODE HERE` region.
- Do not add or delete cells, even empty ones.
- Do not import any library beyond what is already provided. Every part is solvable without one.
- Submit a single zip file, `Roll_Number_q3.zip`, containing your notebook and a single PDF report.
- Your notebook is re-run after the deadline on data it has not seen. If it fails to run top to bottom, no compensation will be given.
- Notebook parts are graded by re-running your code. Report parts are graded by hand, and are where you justify the choices you made in the notebook.

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

PATH_TO_DATA_DIR = "./Q3_DATA"
PATH_TO_TRAIN_FILE = os.path.join(PATH_TO_DATA_DIR, "train.csv")

TARGET_COLUMN = "target"
GROUP_COLUMN = "group_id"
ID_COLUMN = "row_id"

In [ ]:
# Set this to your roll number as an integer. Example: 2024000
ROLL_NUMBER = None
# YOUR CODE HERE

## 1. (1 point) Data Loading

(a) **[1 point]** Load the raw training data so it is ready to explore.

In [ ]:
def load_raw_data(path):
    """
    Load the raw training data.

    Args:
        path (str): Path to the training CSV file.

    Returns:
        pd.DataFrame: The raw data, exactly as it appears in the file.
    """
    raw = None
    # YOUR CODE HERE
    return raw

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

raw_df = load_raw_data(PATH_TO_TRAIN_FILE)
assert raw_df is not None and len(raw_df) > 0, "load_raw_data() did not return a non-empty DataFrame"
print(f"raw_df: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

## 2. (4 points) Exploratory Data Analysis

(a) Explore the raw data. Look at what
kind of features you have, missing values, the target. Put your EDA analysis plots
and tables that mattered in your report. (You would have to justify your choices.)

**Grading:** both parts are graded from your report. Nothing in this
notebook is scored for Question 2 -- this cell is where you explore, freely.

In [ ]:
# YOUR CODE HERE

## 3. (8 points) Data Preparation and Feature Engineering

(a) **[4 point]** Based on the observations from your exploratory data analysis, prepare the dataset for modeling. Clearly explain the preprocessing and feature-
engineering decisions you make and how they are motivated by patterns or
issues identified during EDA. Only short bullet points are needed in the report,
not long explanations.

(b) **[1 point]** Evaluate whether your chosen data preparation and engineered
features are appropriate for modelling. Provide evidence for your choices and
discuss how you ensured that the resulting pipeline remains valid when applied
to previously unseen data

**A submission that fails part (a) or (b) also scores zero on the performance
parts of Questions 5 and 7.**

Model performance based parts are evaluated from rerunning code below, rest are evaluated from report

In [ ]:
class Dataset:
    """Loads a CSV and exposes it as arrays ready for scikit-learn."""

    def __init__(self, filepath, preprocessor=None, do_fit=True):
        self.filepath = filepath
        self.preprocessor = preprocessor
        self.features = None
        self.labels = None
        self.groups = None
        self.feature_names = None
        self._load_and_preprocess(do_fit)

    def _load_and_preprocess(self, do_fit=True):
        """
        Load the CSV and preprocess it into arrays ready for scikit-learn.

        Args:
            do_fit (bool): Whether to fit the preprocessor on this data before
                applying it, or apply an already-fitted one.

        Sets:
            self.features (np.ndarray): Shape (n_samples, n_features), fully
                numeric, no missing values.
            self.labels (np.ndarray or None): Shape (n_samples,), or None if
                the file has no target column.
            self.groups (np.ndarray): Shape (n_samples,).
            self.preprocessor: Fitted, so it can be reused on unseen data.
            self.feature_names (list of str): One name per column of
                self.features.
        """
        # YOUR CODE HERE

    def get_features_and_labels(self):
        return self.features, self.labels

    def get_groups(self):
        return self.groups

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

train_dataset = Dataset(PATH_TO_TRAIN_FILE, do_fit=True)
X_train_full, y_train_full = train_dataset.get_features_and_labels()
print(f"train_dataset: X {X_train_full.shape}, y {y_train_full.shape}, "
      f"groups {len(set(train_dataset.get_groups()))}")

# Part (b): the same preparation, applied unchanged to rows it was not fit on.
# No separate file is provided for this -- a random held-out slice of your own
# training data, refit and re-applied here, is the check.
_check_rng = np.random.default_rng(ROLL_NUMBER)
_holdout_idx = _check_rng.choice(len(raw_df), size=max(10, len(raw_df) // 10), replace=False)
_fit_idx = np.setdiff1d(np.arange(len(raw_df)), _holdout_idx)
_probe_fit_path = "._q3_probe_fit.csv"
_probe_holdout_path = "._q3_probe_holdout.csv"
raw_df.iloc[_fit_idx].to_csv(_probe_fit_path, index=False)
raw_df.iloc[_holdout_idx].to_csv(_probe_holdout_path, index=False)

_probe_fit_dataset = Dataset(_probe_fit_path, do_fit=True)
_probe_holdout_dataset = Dataset(
    _probe_holdout_path, preprocessor=_probe_fit_dataset.preprocessor, do_fit=False
)
_X_holdout = _probe_holdout_dataset.get_features_and_labels()[0]

os.remove(_probe_fit_path)
os.remove(_probe_holdout_path)

assert _X_holdout.shape[1] == _probe_fit_dataset.get_features_and_labels()[0].shape[1], (
    "preparation produced a different number of columns on held-out rows"
)
assert not np.isnan(_X_holdout).any(), "held-out rows have missing values after preparation"
print("preparation generalizes to rows it was not fit on")

# Part (c): row identifiers and grouping information must not reach the model.
assert ID_COLUMN not in train_dataset.feature_names, f"{ID_COLUMN} leaked into the features"
assert not any(GROUP_COLUMN in name for name in train_dataset.feature_names), (
    f"{GROUP_COLUMN} leaked into the features"
)
print("identifier and group columns excluded from the features")

## 4. (3 points) Validation Strategy

(a) Split the training data into a training part and a validation
part.

In [ ]:
def get_validation_split(dataset):
    """
    Split the training data into a training part and a validation part.

    Args:
        dataset (Dataset): The training dataset.

    Returns:
        tuple of np.ndarray: (train_idx, val_idx), integer row indices. Every
            row index must appear in exactly one of the two arrays.
    """
    train_idx, val_idx = None, None
    # YOUR CODE HERE
    return train_idx, val_idx

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

train_idx, val_idx = get_validation_split(train_dataset)
assert len(set(train_idx) & set(val_idx)) == 0, "train and validation indices overlap"
assert len(train_idx) + len(val_idx) == len(y_train_full), "the split does not cover every row"
val_fraction = len(val_idx) / len(y_train_full)
assert 0.10 <= val_fraction <= 0.40, f"validation part is {val_fraction:.0%} of rows, outside 10-40%"
print(f"train rows: {len(train_idx)}, validation rows: {len(val_idx)} ({val_fraction:.0%})")

## 5. (6 points) Model Training and Evaluation

Your score here is computed twice: once in this question, and again in
Question 7 after you iterate on it. The two are graded separately.

(a) **[up to 4 points]** Train a linear regression model on your split. Scored
on a hidden test set:

| R² on hidden test set | Marks |
| :--------------------: | :---: |
| ≥ 0.90 | 4 |
| 0.85 to 0.90 | 3 |
| 0.75 to 0.85 | 2 |
| 0.65 to 0.75 | 1 |
| below 0.65 | 0 |

(b) **[2 points]** Justify your choice of model and hyperparameters. Before
looking at Question 6's diagnostics, say what you expect this choice to do to
the train and validation gap.

The cell below trains your model and reports its performance on your OWN
train/validation split -- use these numbers to decide what to submit. The
hidden-set score in the table above is computed separately after submission.

In [ ]:
def get_regressor():
    """Return an unfitted sklearn.linear_model regressor with
    student-selected hyperparameters."""
    model = None
    # YOUR CODE HERE
    return model

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

class ModelTrainer:
    """Fits, evaluates and predicts with a scikit-learn regressor, taking
    Dataset objects directly."""

    def __init__(self, regressor):
        if not all(hasattr(regressor, attr) for attr in ("fit", "predict")):
            raise TypeError("The provided object is not a valid sklearn regressor.")
        self.regressor = regressor
        self.is_trained = False

    def train(self, dataset, indices=None):
        X, y = dataset.get_features_and_labels()
        if y is None:
            raise ValueError("Cannot train on a dataset with no labels.")
        if indices is not None:
            X, y = X[indices], y[indices]
        self.regressor.fit(X, y)
        self.is_trained = True
        return self

    def predict(self, dataset, indices=None):
        if not self.is_trained:
            raise ValueError("Regressor has not been trained. Call .train() first.")
        X, _ = dataset.get_features_and_labels()
        if indices is not None:
            X = X[indices]
        return np.clip(self.regressor.predict(X), 0, None)

    def evaluate(self, dataset, indices=None, title_prefix=""):
        y = dataset.get_features_and_labels()[1]
        if y is None:
            raise ValueError("Cannot evaluate on a dataset with no labels.")
        if indices is not None:
            y = y[indices]
        pred = self.predict(dataset, indices)

        scores = {
            "R2": r2_score(y, pred),
            "RMSE": mean_squared_error(y, pred) ** 0.5,
            "MAE": mean_absolute_error(y, pred),
        }
        header = f"{title_prefix} {self.regressor.__class__.__name__} Results".strip()
        print(header)
        print("-" * len(header))
        for key, value in scores.items():
            print(f"{key:5s}: {value:.4f}")
        return scores

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

trainer = ModelTrainer(get_regressor()).train(train_dataset, train_idx)
train_scores = trainer.evaluate(train_dataset, train_idx, title_prefix="[Train]")
print()
val_scores = trainer.evaluate(train_dataset, val_idx, title_prefix="[Validation]")

## 6. (6 points) Diagnostics

(a) **[2 points]** Produce diagnostics for your trained model: the gap between
train and validation performance, a learning curve for both, and the rows it
gets most wrong.

(b) **[2 points]** Look at the train and validation gap and the learning curve.
Say what they tell you about your model.

(c) **[2 points]** Look at the rows your model gets most wrong. Say what they
have in common, and what you would try next.

**Grading:** part (a) is graded by re-running the cell below -- it runs on the
model you already trained in Question 5, not a fresh one. Parts (b) and (c) are
graded from your report.

In [ ]:
class Diagnostics:
    """Diagnostics for an already-trained model."""

    @staticmethod
    def bias_variance_report(trainer, dataset, train_idx, val_idx):
        """
        Report train and validation performance for a trained model.

        Args:
            trainer (ModelTrainer): An already-trained model.
            dataset (Dataset): The training dataset.
            train_idx, val_idx (np.ndarray): Row indices from Question 4.

        Returns:
            dict: {"train": scores, "validation": scores, "gap": float}, where
                "train"/"validation" are the dicts returned by
                trainer.evaluate(), and "gap" is train R2 minus validation R2.
        """
        result = None
        # YOUR CODE HERE
        return result

    @staticmethod
    def learning_curve(dataset, train_idx, val_idx, regressor_factory,
                        fractions=(0.1, 0.25, 0.5, 0.75, 1.0)):
        """
        Train on increasing fractions of train_idx and report R2 on each
        fraction and on val_idx, so train/validation performance can be
        compared as training set size grows.

        Args:
            dataset (Dataset): The training dataset.
            train_idx, val_idx (np.ndarray): Row indices from Question 4.
            regressor_factory (callable): Returns a fresh unfitted regressor,
                e.g. get_regressor.
            fractions (tuple of float): Fractions of train_idx to train on.

        Returns:
            dict: {"sizes": [...], "train_r2": [...], "validation_r2": [...]},
                one entry per fraction, same order as `fractions`.
        """
        result = None
        # YOUR CODE HERE
        return result

    @staticmethod
    def error_analysis(trainer, dataset, val_idx, k=15):
        """
        Report the k validation rows the model gets most wrong.

        Args:
            trainer (ModelTrainer): An already-trained model.
            dataset (Dataset): The training dataset.
            val_idx (np.ndarray): Validation row indices from Question 4.
            k (int): Number of worst rows to report.

        Returns:
            pd.DataFrame: k rows sorted by absolute residual, descending, with
                columns row, group, y_true, y_pred, residual.
        """
        table = None
        # YOUR CODE HERE
        return table

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

# Runs on `trainer`, already trained in Question 5 -- nothing here retrains.
report = Diagnostics.bias_variance_report(trainer, train_dataset, train_idx, val_idx)
curve = Diagnostics.learning_curve(train_dataset, train_idx, val_idx, get_regressor)
errors = Diagnostics.error_analysis(trainer, train_dataset, val_idx)

## 7. (9 points) Model Improvement

(a) **[up to 4 points]** Take what you learned from Question 6 and change
something: your model, its hyperparameters, or how you are using your
features. Retrain. Scored on the same hidden test set as Question 5:

| R² on hidden test set | Marks |
| :--------------------: | :---: |
| ≥ 0.90 | 4 |
| 0.85 to 0.90 | 3 |
| 0.75 to 0.85 | 2 |
| 0.65 to 0.75 | 1 |
| below 0.65 | 0 |

(b) **[3 points]** Explain what you changed and why, based on your diagnostics.

(c) **[2 points]** Compare this score with Question 5's, and say whether the
change helped.

**Grading:** part (a) is graded by re-running the cell below. Parts (b) and (c)
are graded from your report.

In [ ]:
def get_improved_regressor():
    """Return an unfitted sklearn.linear_model regressor with
    student-selected hyperparameters."""
    model = None
    # YOUR CODE HERE
    return model

In [ ]:
# =======================
# DO NOT MODIFY THIS CELL
# =======================

final_trainer = ModelTrainer(get_improved_regressor()).train(train_dataset, train_idx)
final_trainer.evaluate(train_dataset, train_idx, title_prefix="[Train]")
print()
final_trainer.evaluate(train_dataset, val_idx, title_prefix="[Validation]")

## 8. (3 points) Generalization and Limitations

(a) **[1 point]** The hidden test set contains groups you have never seen. Give
one concrete way your pipeline could still fail on them even with a good score.

(b) **[2 points]** Name one change you would try next, to the data, the
features, the model, or the validation strategy, and explain why you expect it
to help the most.

**Grading:** both parts are graded from your report. Nothing to run
here.